# Notebook 02: First Cell Locator

## Goal
Find the top-left-most cell/blob inside the detected Sudoku border.

## Concepts Covered
1. **Perspective transform** - Warp quadrilateral to rectangle (homography)
2. **Connected component analysis** - Label and analyze distinct regions
3. **Blob filtering** - Filter by area, aspect ratio, and shape
4. **Coordinate geometry** - Find extremal points in a set

## Algorithm
```
1. Apply perspective transform using corners from Notebook 01
2. Preprocess warped image: threshold to binary
3. Invert so cell interiors are white (foreground)
4. Run connected components analysis
5. Filter blobs by area and aspect ratio
6. Find blob with minimum (centroid_x + centroid_y) → top-left cell
```

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

# Our utilities
from utils.visualization import (
    show_image, show_images_grid, draw_quadrilateral, 
    draw_corners, draw_cell_highlight
)
from utils.geometry import order_corners, compute_quad_area

# Sample image loading
from tests.border_detection.sampler import sample_images, load_image

print(f"Project root: {PROJECT_ROOT}")

## Load Sample Image and Detect Border

First, we'll use the border detection from Notebook 01 to get the grid corners.

In [ ]:
# Functions from Notebook 01
def find_outer_border(image, blur_kernel=5, threshold_block_size=11, threshold_c=2):
    """Detect outer border using contour detection."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur_kernel, blur_kernel), 0)
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, threshold_block_size, threshold_c
    )
    
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    for contour in sorted(contours, key=cv2.contourArea, reverse=True):
        peri = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
        if len(approx) == 4:
            return order_corners(approx.reshape(4, 2))
    
    return None

# Load sample image
sample_paths = sample_images(1, seed=42)
image_path = sample_paths[0]
original = load_image(image_path)

# Detect border
corners = find_outer_border(original)

if corners is not None:
    print(f"Loaded: {image_path.name}")
    print(f"Detected corners:\n{corners}")
    
    # Visualize
    vis = draw_quadrilateral(original, corners)
    vis = draw_corners(vis, corners)
    show_image(vis, "Detected Border")
else:
    print("Failed to detect border!")

---
## Step 1: Perspective Transform

### What is Perspective Transform?
- Maps a quadrilateral to a rectangle (or vice versa)
- Uses a **homography matrix** (3x3) to define the transformation
- Corrects for camera angle and perspective distortion

### The Math
A homography maps points from one plane to another:
```
[x']   [h11 h12 h13]   [x]
[y'] = [h21 h22 h23] * [y]
[w']   [h31 h32 h33]   [1]

x_out = x' / w'
y_out = y' / w'
```

OpenCV's `getPerspectiveTransform` computes this 3x3 matrix from 4 point correspondences.

In [ ]:
def warp_to_top_down(
    image: np.ndarray,
    corners: np.ndarray,
    output_size: int = 450,
) -> np.ndarray:
    """
    Apply perspective transform to get a top-down view of the grid.
    
    Args:
        image: Input image (BGR)
        corners: 4x2 array of source corners [TL, TR, BR, BL]
        output_size: Size of output square image
        
    Returns:
        Warped square image of size (output_size, output_size)
    """
    # Source points (detected corners)
    src_pts = corners.astype(np.float32)
    
    # Destination points (square)
    dst_pts = np.array([
        [0, 0],                    # TL
        [output_size - 1, 0],      # TR
        [output_size - 1, output_size - 1],  # BR
        [0, output_size - 1],      # BL
    ], dtype=np.float32)
    
    # Compute transformation matrix
    matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
    
    # Apply transformation
    warped = cv2.warpPerspective(image, matrix, (output_size, output_size))
    
    return warped

# Apply perspective transform
if corners is not None:
    warped = warp_to_top_down(original, corners, output_size=450)
    
    print(f"Original shape: {original.shape}")
    print(f"Warped shape: {warped.shape}")
    
    show_images_grid(
        [original, warped],
        ["Original (with perspective)", "Warped (top-down view)"],
        cols=2
    )

---
## Step 2: Preprocess for Blob Detection

### Goal
Create a binary image where:
- Cell interiors are **white** (255) - these are our blobs
- Grid lines are **black** (0) - these separate the cells

### Strategy
1. Convert to grayscale
2. Apply adaptive threshold (cells become black, background white)
3. **Invert** so cells are white

In [ ]:
def preprocess_for_blobs(
    image: np.ndarray,
    block_size: int = 11,
    c: int = 2,
) -> np.ndarray:
    """
    Preprocess image for blob detection.
    
    Creates a binary image where cell interiors are white.
    
    Args:
        image: Input BGR image
        block_size: Adaptive threshold block size
        c: Adaptive threshold constant
        
    Returns:
        Binary image with white cells on black background
    """
    # Convert to grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Adaptive threshold - grid lines become white, cells black
    binary = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        block_size, c
    )
    
    # Invert so cells are white
    inverted = cv2.bitwise_not(binary)
    
    return inverted

# Preprocess
if corners is not None:
    binary_cells = preprocess_for_blobs(warped)
    
    show_images_grid(
        [warped, binary_cells],
        ["Warped Grid", "Binary (cells = white)"],
        cols=2
    )

---
## Step 3: Connected Components Analysis

### What are Connected Components?
- Groups of connected pixels with the same value
- In a binary image, each isolated white region is a component
- Components are **labeled** with unique integers (1, 2, 3, ...)
- Label 0 is reserved for background (black pixels)

### OpenCV's `connectedComponentsWithStats`
Returns:
- `num_labels`: Total number of labels (including background)
- `labels`: Image where each pixel has its component label
- `stats`: Array with statistics for each component:
  - `cv2.CC_STAT_LEFT`: Leftmost x coordinate
  - `cv2.CC_STAT_TOP`: Topmost y coordinate
  - `cv2.CC_STAT_WIDTH`: Width of bounding box
  - `cv2.CC_STAT_HEIGHT`: Height of bounding box
  - `cv2.CC_STAT_AREA`: Area in pixels
- `centroids`: (x, y) centroid for each component

In [ ]:
@dataclass
class BlobInfo:
    """Information about a detected blob."""
    label: int
    x: int       # Left edge
    y: int       # Top edge
    width: int
    height: int
    area: int
    centroid_x: float
    centroid_y: float
    
    @property
    def aspect_ratio(self) -> float:
        """Width / Height ratio."""
        return self.width / self.height if self.height > 0 else 0
    
    @property
    def bbox(self) -> Tuple[int, int, int, int]:
        """Bounding box as (x, y, width, height)."""
        return (self.x, self.y, self.width, self.height)
    
    @property
    def centroid(self) -> Tuple[float, float]:
        """Centroid as (x, y)."""
        return (self.centroid_x, self.centroid_y)


def find_connected_components(binary: np.ndarray) -> List[BlobInfo]:
    """
    Find all connected components (blobs) in a binary image.
    
    Args:
        binary: Binary image (white foreground)
        
    Returns:
        List of BlobInfo objects for each component (excluding background)
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
    
    blobs = []
    
    # Skip label 0 (background)
    for i in range(1, num_labels):
        blob = BlobInfo(
            label=i,
            x=stats[i, cv2.CC_STAT_LEFT],
            y=stats[i, cv2.CC_STAT_TOP],
            width=stats[i, cv2.CC_STAT_WIDTH],
            height=stats[i, cv2.CC_STAT_HEIGHT],
            area=stats[i, cv2.CC_STAT_AREA],
            centroid_x=centroids[i, 0],
            centroid_y=centroids[i, 1],
        )
        blobs.append(blob)
    
    return blobs

# Find all blobs
if corners is not None:
    all_blobs = find_connected_components(binary_cells)
    
    print(f"Found {len(all_blobs)} connected components")
    print("\nFirst 5 blobs:")
    for blob in all_blobs[:5]:
        print(f"  Label {blob.label}: area={blob.area}, "
              f"aspect={blob.aspect_ratio:.2f}, "
              f"centroid=({blob.centroid_x:.1f}, {blob.centroid_y:.1f})")

In [ ]:
# Visualize all blobs with bounding boxes
def visualize_blobs(image: np.ndarray, blobs: List[BlobInfo], max_blobs: int = 50) -> np.ndarray:
    """Draw bounding boxes around detected blobs."""
    vis = image.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    for blob in blobs[:max_blobs]:
        color = (0, 255, 0)
        cv2.rectangle(
            vis,
            (blob.x, blob.y),
            (blob.x + blob.width, blob.y + blob.height),
            color, 1
        )
        # Draw centroid
        cx, cy = int(blob.centroid_x), int(blob.centroid_y)
        cv2.circle(vis, (cx, cy), 3, (0, 0, 255), -1)
    
    return vis

if corners is not None:
    blob_vis = visualize_blobs(warped, all_blobs)
    show_image(blob_vis, f"All Blobs ({len(all_blobs)} found)")

---
## Step 4: Filter Blobs by Size and Shape

### Why Filter?
Not all blobs are cells:
- Very small blobs = noise
- Very large blobs = merged cells or background
- Non-square blobs = partial cells or artifacts

### Filtering Criteria
For a 450x450 warped grid with 9x9 cells:
- Expected cell size: ~50x50 pixels
- Expected cell area: ~2500 pixels
- Area range: 1000 to 5000 (allowing for variations)
- Aspect ratio: 0.5 to 2.0 (roughly square)

In [ ]:
def filter_cell_blobs(
    blobs: List[BlobInfo],
    image_size: int,
    min_area_ratio: float = 0.003,  # Min area as fraction of image
    max_area_ratio: float = 0.02,   # Max area as fraction of image
    min_aspect: float = 0.5,
    max_aspect: float = 2.0,
) -> List[BlobInfo]:
    """
    Filter blobs to keep only cell-like ones.
    
    Args:
        blobs: List of all detected blobs
        image_size: Size of the warped image (assumed square)
        min_area_ratio: Minimum blob area / image area
        max_area_ratio: Maximum blob area / image area
        min_aspect: Minimum aspect ratio (width/height)
        max_aspect: Maximum aspect ratio
        
    Returns:
        Filtered list of cell-like blobs
    """
    image_area = image_size * image_size
    min_area = min_area_ratio * image_area
    max_area = max_area_ratio * image_area
    
    filtered = []
    
    for blob in blobs:
        # Check area
        if blob.area < min_area or blob.area > max_area:
            continue
        
        # Check aspect ratio
        if blob.aspect_ratio < min_aspect or blob.aspect_ratio > max_aspect:
            continue
        
        filtered.append(blob)
    
    return filtered

# Filter blobs
if corners is not None:
    cell_blobs = filter_cell_blobs(all_blobs, image_size=450)
    
    print(f"After filtering: {len(cell_blobs)} cell-like blobs (from {len(all_blobs)})")
    
    # Show area distribution
    areas = [b.area for b in cell_blobs]
    if areas:
        print(f"Area range: {min(areas)} - {max(areas)}")
        print(f"Mean area: {np.mean(areas):.0f}")

In [ ]:
# Visualize filtered blobs
if corners is not None and cell_blobs:
    filtered_vis = visualize_blobs(warped, cell_blobs)
    show_images_grid(
        [visualize_blobs(warped, all_blobs), filtered_vis],
        [f"All Blobs ({len(all_blobs)})", f"Filtered Cell Blobs ({len(cell_blobs)})"],
        cols=2
    )

---
## Step 5: Find Top-Left Cell

### The Algorithm
The top-left cell has:
- The smallest sum of coordinates (centroid_x + centroid_y)

This is the same principle we used for corner ordering!

### Why Centroid?
Using the centroid is more robust than using the bounding box corner:
- Centroid is at the geometric center of the blob
- Less affected by irregular blob boundaries

In [ ]:
def find_top_left_cell(blobs: List[BlobInfo]) -> Optional[BlobInfo]:
    """
    Find the top-left-most cell (minimum centroid_x + centroid_y).
    
    Args:
        blobs: List of filtered cell blobs
        
    Returns:
        The top-left cell, or None if no blobs
    """
    if not blobs:
        return None
    
    # Find blob with minimum sum of centroid coordinates
    return min(blobs, key=lambda b: b.centroid_x + b.centroid_y)

# Find top-left cell
if corners is not None and cell_blobs:
    top_left_cell = find_top_left_cell(cell_blobs)
    
    if top_left_cell:
        print(f"Top-left cell found!")
        print(f"  Position: ({top_left_cell.x}, {top_left_cell.y})")
        print(f"  Size: {top_left_cell.width} x {top_left_cell.height}")
        print(f"  Centroid: ({top_left_cell.centroid_x:.1f}, {top_left_cell.centroid_y:.1f})")
        print(f"  Centroid sum: {top_left_cell.centroid_x + top_left_cell.centroid_y:.1f}")

In [ ]:
# Visualize the top-left cell
if corners is not None and top_left_cell:
    # Create cell corners from bounding box
    x, y, w, h = top_left_cell.bbox
    cell_corners = np.array([
        [x, y],
        [x + w, y],
        [x + w, y + h],
        [x, y + h],
    ])
    
    # Draw on warped image
    result = draw_cell_highlight(
        warped, cell_corners,
        fill_color=(0, 255, 255),
        fill_alpha=0.4,
        border_color=(0, 255, 0),
        border_thickness=2
    )
    
    # Mark centroid
    cx, cy = int(top_left_cell.centroid_x), int(top_left_cell.centroid_y)
    cv2.circle(result, (cx, cy), 5, (0, 0, 255), -1)
    cv2.putText(result, "TOP-LEFT", (x, y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    show_image(result, "Top-Left Cell Detected")

---
## Complete Function: `find_first_cell`

Let's combine all steps into a single reusable function.

In [ ]:
@dataclass
class CellDetectionResult:
    """Result from first cell detection."""
    success: bool
    cell: Optional[BlobInfo] = None
    warped_image: Optional[np.ndarray] = None
    all_blobs: Optional[List[BlobInfo]] = None
    filtered_blobs: Optional[List[BlobInfo]] = None
    debug_images: Optional[Dict[str, np.ndarray]] = None


def find_first_cell(
    image: np.ndarray,
    corners: np.ndarray,
    output_size: int = 450,
    threshold_block_size: int = 11,
    threshold_c: int = 2,
    min_area_ratio: float = 0.003,
    max_area_ratio: float = 0.02,
    return_debug: bool = False,
) -> CellDetectionResult:
    """
    Find the top-left cell in a Sudoku grid.
    
    Args:
        image: Input BGR image
        corners: 4x2 array of grid corners [TL, TR, BR, BL]
        output_size: Size of warped image
        threshold_block_size: Adaptive threshold block size
        threshold_c: Adaptive threshold constant
        min_area_ratio: Minimum blob area ratio
        max_area_ratio: Maximum blob area ratio
        return_debug: Whether to return intermediate images
        
    Returns:
        CellDetectionResult with cell info and optional debug images
    """
    debug = {}
    
    # Step 1: Perspective transform
    warped = warp_to_top_down(image, corners, output_size)
    if return_debug:
        debug["1_warped"] = warped
    
    # Step 2: Preprocess for blobs
    binary = preprocess_for_blobs(warped, threshold_block_size, threshold_c)
    if return_debug:
        debug["2_binary"] = binary
    
    # Step 3: Find connected components
    all_blobs = find_connected_components(binary)
    if return_debug:
        debug["3_all_blobs"] = visualize_blobs(warped, all_blobs)
    
    # Step 4: Filter blobs
    cell_blobs = filter_cell_blobs(
        all_blobs, output_size, min_area_ratio, max_area_ratio
    )
    if return_debug:
        debug["4_filtered"] = visualize_blobs(warped, cell_blobs)
    
    # Step 5: Find top-left cell
    top_left = find_top_left_cell(cell_blobs)
    
    if top_left is None:
        return CellDetectionResult(
            success=False,
            warped_image=warped,
            all_blobs=all_blobs,
            filtered_blobs=cell_blobs,
            debug_images=debug if return_debug else None,
        )
    
    if return_debug:
        x, y, w, h = top_left.bbox
        cell_corners = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]])
        result_vis = draw_cell_highlight(warped, cell_corners)
        debug["5_result"] = result_vis
    
    return CellDetectionResult(
        success=True,
        cell=top_left,
        warped_image=warped,
        all_blobs=all_blobs,
        filtered_blobs=cell_blobs,
        debug_images=debug if return_debug else None,
    )

In [ ]:
# Test the complete function
if corners is not None:
    result = find_first_cell(original, corners, return_debug=True)
    
    print(f"Success: {result.success}")
    if result.success:
        print(f"Cell centroid: ({result.cell.centroid_x:.1f}, {result.cell.centroid_y:.1f})")
        print(f"Cell size: {result.cell.width} x {result.cell.height}")
    
    # Show debug images
    if result.debug_images:
        show_images_grid(
            list(result.debug_images.values()),
            list(result.debug_images.keys()),
            cols=3
        )

---
## Test on Multiple Images

In [ ]:
# Test on 6 sample images
test_paths = sample_images(6, seed=123)
test_images = [load_image(p) for p in test_paths]

result_images = []
success_count = 0

for i, (path, img) in enumerate(zip(test_paths, test_images)):
    # Detect border
    border = find_outer_border(img)
    
    if border is None:
        result_images.append(img.copy())
        continue
    
    # Find first cell
    result = find_first_cell(img, border)
    
    if result.success:
        success_count += 1
        x, y, w, h = result.cell.bbox
        cell_corners = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]])
        vis = draw_cell_highlight(result.warped_image, cell_corners)
        cv2.putText(vis, "TOP-LEFT", (x, y - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    else:
        vis = result.warped_image if result.warped_image is not None else img.copy()
        cv2.putText(vis, "FAILED", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    
    result_images.append(vis)

print(f"Success rate: {success_count}/{len(test_images)}")

titles = [f"{p.name[:15]}..." for p in test_paths]
show_images_grid(result_images, titles, cols=3)

---
## Extracting Cell Information

The detected cell gives us important information for the next step:
- **Cell size** (width, height) - approximate size of each cell
- **Cell position** - where the first cell is located
- **Orientation** - cells should be roughly aligned with axes after warping

In [ ]:
if corners is not None and result.success:
    cell = result.cell
    
    print("First Cell Properties:")
    print(f"  Bounding Box: x={cell.x}, y={cell.y}, w={cell.width}, h={cell.height}")
    print(f"  Area: {cell.area} px²")
    print(f"  Centroid: ({cell.centroid_x:.1f}, {cell.centroid_y:.1f})")
    print(f"  Aspect Ratio: {cell.aspect_ratio:.2f}")
    
    # Estimated cell size for the full grid
    estimated_cell_size = (cell.width + cell.height) / 2
    print(f"\nEstimated cell size: {estimated_cell_size:.1f} px")
    print(f"Expected for 450px grid: {450/9:.1f} px")

---
## Summary

### What We Learned
1. **Perspective transform** warps a quadrilateral to a rectangle using homography
2. **Connected components** labels and analyzes isolated regions in binary images
3. **Blob filtering** by area and aspect ratio removes noise and non-cell regions
4. **Top-left finding** uses coordinate sum to identify position

### Key Parameters
| Parameter | Default | Purpose |
|-----------|---------|--------|
| `output_size` | 450 | Warped image size (multiple of 9) |
| `min_area_ratio` | 0.003 | Filter small noise |
| `max_area_ratio` | 0.02 | Filter large blobs |
| `min/max_aspect` | 0.5-2.0 | Keep roughly square blobs |

### Output for Next Step
- **Top-left cell bounding box**: Starting point for edge detection
- **Estimated cell size**: Guide for local Hough parameters
- **Warped image**: Clean, normalized grid for further processing

### Next Steps
In **Notebook 03**, we'll:
- Extract a Region of Interest (ROI) around the first cell
- Apply **local Hough transform** to detect cell edges
- Find the precise cell boundaries using short line segments